In [1]:
!pip show chromadb rank_bm25 sentence-transformers | grep -E "Name|Version"

Name: sentence-transformers
Version: 5.4.0


In [2]:
!pip install -q chromadb rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [7]:
import os
import shutil

src = '/kaggle/input/researchlens-dataset'
dst = '/kaggle/working'

os.makedirs(dst, exist_ok=True)

for fname in ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl', 
              'semantic_chunks.pkl', 'parsed_papers.json', 
              'papers_metadata.json', 'qasper_pairs.json']:
    src_path = os.path.join(src, fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(dst, fname))
        print(f"  ✓ {fname}")

chroma_src = os.path.join(src, 'chroma_db')
chroma_dst = os.path.join(dst, 'chroma_db')
if os.path.exists(chroma_src) and not os.path.exists(chroma_dst):
    shutil.copytree(chroma_src, chroma_dst)
    print(f"  ✓ chroma_db/")

print("\nDone loading from dataset")


Done loading from dataset


In [8]:
import pickle
import numpy as np
from abc import ABC, abstractmethod
from typing import List, Dict
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [9]:
with open('/kaggle/working/bm25_index.pkl', 'rb') as f:
    bm25_index = pickle.load(f)

with open('/kaggle/working/bm25_metadata.pkl', 'rb') as f:
    bm25_metadata = pickle.load(f)

with open('/kaggle/working/fixed_chunks.pkl', 'rb') as f:
    fixed_chunks = pickle.load(f)

with open('/kaggle/working/semantic_chunks.pkl', 'rb') as f:
    semantic_chunks = pickle.load(f)

chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')
fixed_collection = chroma_client.get_collection("fixed_chunks")
semantic_collection = chroma_client.get_collection("semantic_chunks")

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"BM25 index: {len(bm25_metadata)} docs")
print(f"ChromaDB fixed: {fixed_collection.count()} docs")
print(f"ChromaDB semantic: {semantic_collection.count()} docs")
print(f"Embed model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BM25 index: 3205 docs
ChromaDB fixed: 3205 docs
ChromaDB semantic: 3427 docs
Embed model loaded


In [10]:
class BaseRetriever(ABC):
    @abstractmethod
    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        pass

    @property
    @abstractmethod
    def strategy_name(self) -> str:
        pass

    def _make_result(self, text, score, paper_id, title, chunk_id):
        return {
            'text': text,
            'score': float(score),
            'paper_id': paper_id,
            'title': title,
            'chunk_id': chunk_id
        }

In [11]:
class NaiveRetriever(BaseRetriever):
    @property
    def strategy_name(self):
        return "naive_rag"

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        query_embedding = embed_model.encode(query).tolist()

        results = fixed_collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=['documents', 'distances', 'metadatas']
        )

        output = []
        for doc, dist, meta in zip(
            results['documents'][0],
            results['distances'][0],
            results['metadatas'][0]
        ):
            output.append(self._make_result(
                text=doc,
                score=1 - dist,
                paper_id=meta['paper_id'],
                title=meta['title'],
                chunk_id=meta.get('chunk_id', '')
            ))

        return output

In [12]:
class SemanticRetriever(BaseRetriever):
    @property
    def strategy_name(self):
        return "semantic_rag"

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        query_embedding = embed_model.encode(query).tolist()

        results = semantic_collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=['documents', 'distances', 'metadatas']
        )

        output = []
        for doc, dist, meta in zip(
            results['documents'][0],
            results['distances'][0],
            results['metadatas'][0]
        ):
            output.append(self._make_result(
                text=doc,
                score=1 - dist,
                paper_id=meta['paper_id'],
                title=meta['title'],
                chunk_id=meta.get('chunk_id', '')
            ))

        return output

In [14]:
class HybridRetriever(BaseRetriever):
    @property
    def strategy_name(self):
        return "hybrid_rag"

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        fetch_k = top_k * 4

        query_embedding = embed_model.encode(query).tolist()
        dense_results = fixed_collection.query(
            query_embeddings=[query_embedding],
            n_results=fetch_k,
            include=['documents', 'distances', 'metadatas']
        )

        dense_hits = {}
        for rank, (doc, dist, meta) in enumerate(zip(
            dense_results['documents'][0],
            dense_results['distances'][0],
            dense_results['metadatas'][0]
        )):
            chunk_id = meta.get('chunk_id', f"dense_{rank}")
            dense_hits[chunk_id] = {
                'rank': rank,
                'text': doc,
                'score': 1 - dist,
                'paper_id': meta['paper_id'],
                'title': meta['title']
            }

        bm25_scores = bm25_index.get_scores(query.lower().split())
        top_bm25_idx = np.argsort(bm25_scores)[::-1][:fetch_k]

        bm25_hits = {}
        for rank, idx in enumerate(top_bm25_idx):
            chunk_id = bm25_metadata[idx]['chunk_id']
            bm25_hits[chunk_id] = {
                'rank': rank,
                'text': bm25_metadata[idx]['text'],
                'score': bm25_scores[idx],
                'paper_id': bm25_metadata[idx]['paper_id'],
                'title': bm25_metadata[idx]['title']
            }

        k = 60
        all_chunk_ids = set(dense_hits.keys()) | set(bm25_hits.keys())
        rrf_scores = {}

        for chunk_id in all_chunk_ids:
            rrf = 0.0
            if chunk_id in dense_hits:
                rrf += 1 / (dense_hits[chunk_id]['rank'] + k)
            if chunk_id in bm25_hits:
                rrf += 1 / (bm25_hits[chunk_id]['rank'] + k)
            rrf_scores[chunk_id] = rrf

        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

        output = []
        for chunk_id in top_ids:
            source = dense_hits.get(chunk_id) or bm25_hits.get(chunk_id)
            output.append(self._make_result(
                text=source['text'],
                score=rrf_scores[chunk_id],
                paper_id=source['paper_id'],
                title=source['title'],
                chunk_id=chunk_id
            ))

        return output

In [15]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded


In [16]:
class HybridRerankerRetriever(BaseRetriever):
    @property
    def strategy_name(self):
        return "hybrid_rerank"

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        rerank_pool = 20
        hybrid = HybridRetriever()
        candidates = hybrid.retrieve(query, top_k=rerank_pool)

        pairs = [[query, c['text']] for c in candidates]
        rerank_scores = reranker.predict(pairs)

        for candidate, score in zip(candidates, rerank_scores):
            candidate['score'] = float(score)

        reranked = sorted(candidates, key=lambda x: x['score'], reverse=True)

        return reranked[:top_k]

In [17]:
def compare_strategies(query: str, top_k: int = 3):
    strategies = [
        NaiveRetriever(),
        SemanticRetriever(),
        HybridRetriever(),
        HybridRerankerRetriever()
    ]

    print(f"Query: '{query}'")
    print("=" * 80)

    for strategy in strategies:
        print(f"\n--- {strategy.strategy_name.upper()} ---")
        results = strategy.retrieve(query, top_k=top_k)
        for i, r in enumerate(results):
            print(f"  [{i+1}] Score: {r['score']:.4f} | {r['title'][:55]}")
            print(f"       {r['text'][:180]}")

test_queries = [
    "What is the difference between LoRA and full fine-tuning?",
    "How does attention mechanism work in transformers?",
    "What are the limitations of retrieval augmented generation?"
]

for query in test_queries:
    compare_strategies(query)
    print("\n" + "=" * 80 + "\n")

Query: 'What is the difference between LoRA and full fine-tuning?'

--- NAIVE_RAG ---
  [1] Score: 0.6100 | Theoretical Insights into Fine-Tuning Attention Mechani
       small number of parameters of the LLM such as LoRA . Furthermore, He et al. present a unified framework that establishes connections between PEFT methods and Zhu et al. formally id
  [2] Score: 0.5995 | Task-Specific Directions: Definition, Exploration, and 
       in all the experiments, and the detailed experimental settings are shown in Secs. 8.2-8.4, respectively. 5.2 Numerical Results The numerical results, as shown in Tables. 1-2, demon
  [3] Score: 0.5726 | Task-Specific Directions: Definition, Exploration, and 
       A SUBMISSION TO IEEE TRANSACTION ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE 22 standard LoRA. This overhead mainly arises from the SVD factorization of the pre-trained weight mat

--- SEMANTIC_RAG ---
  [1] Score: 0.5726 | Task-Specific Directions: Definition, Exploration, and 
       A SUBMISS

In [18]:
import time

def benchmark_latency(query: str, runs: int = 3):
    strategies = [
        NaiveRetriever(),
        SemanticRetriever(),
        HybridRetriever(),
        HybridRerankerRetriever()
    ]

    print(f"Latency benchmark (avg of {runs} runs)")
    print(f"Query: '{query[:60]}'")
    print("-" * 40)

    latencies = {}
    for strategy in strategies:
        times = []
        for _ in range(runs):
            start = time.time()
            strategy.retrieve(query, top_k=5)
            times.append(time.time() - start)
        avg = np.mean(times)
        latencies[strategy.strategy_name] = avg
        print(f"  {strategy.strategy_name:<20} {avg*1000:.0f}ms")

    return latencies

latencies = benchmark_latency("What is retrieval augmented generation?")

Latency benchmark (avg of 3 runs)
Query: 'What is retrieval augmented generation?'
----------------------------------------
  naive_rag            11ms
  semantic_rag         8ms
  hybrid_rag           16ms
  hybrid_rerank        166ms


In [19]:
import json

with open('/kaggle/working/latency_results.json', 'w') as f:
    json.dump(latencies, f)

print("Saved latency_results.json")
print("\n=== Notebook 03 Complete ===")
print("Strategies implemented: NaiveRetriever, SemanticRetriever, HybridRetriever, HybridRerankerRetriever")

Saved latency_results.json

=== Notebook 03 Complete ===
Strategies implemented: NaiveRetriever, SemanticRetriever, HybridRetriever, HybridRerankerRetriever
